INSIGHT GENERATION

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("amazon_prime_titles.csv")

df.head()

In [ ]:
prime = pd.read_csv("amazon_prime_titles.csv")

# Helper columns used in the insights
prime["duration_value"] = pd.to_numeric(prime["duration"].str.extract(r"(\d+)")[0])
prime["runtime_minutes"] = prime["duration_value"].where(prime["type"] == "Movie")
prime["seasons"] = prime["duration_value"].where(prime["type"] == "TV Show")
prime["rating"] = prime["rating"].fillna(prime["rating"].mode()[0])
prime["country"] = prime["country"].fillna("Unknown")

In [ ]:
titles_by_country = prime[prime["country"] != "Unknown"]["country"].value_counts()

print("Top 10 countries by number of titles (known country only):")
print(titles_by_country.head(10))

In [ ]:
print(prime.columns.tolist())

In [ ]:
prime.groupby("type")["release_year"].mean()

Insights by content type (Movie vs TV Show)

In [ ]:
type_insights = prime.groupby("type").agg(
    number_of_titles=("title", "count"),
    average_release_year=("release_year", "mean"),
    average_runtime_minutes=("runtime_minutes", "mean"),
    average_seasons=("seasons", "mean")
)

type_insights.round(2)

Insights by genre (titles list several genres, so each genre is counted separately)

In [ ]:
genre_df = prime.assign(genre=prime["listed_in"].str.split(", ")).explode("genre")

genre_insights = genre_df.groupby("genre").agg(
    number_of_titles=("title", "count"),
    average_release_year=("release_year", "mean")
)

genre_insights_30 = genre_insights[genre_insights["number_of_titles"] >= 30]

genre_insights_30.sort_values("number_of_titles", ascending=False).round(2).head(15)

Insights by age rating

In [ ]:
rating_insights = prime.groupby("rating").agg(
    number_of_titles=("title", "count"),
    average_release_year=("release_year", "mean"),
    average_runtime_minutes=("runtime_minutes", "mean")
)

rating_insights.sort_values("number_of_titles", ascending=False).round(2)

Insights by release decade

In [ ]:
prime["decade"] = prime["release_year"] // 10 * 10

decade_insights = prime.groupby("decade").agg(
    number_of_titles=("title", "count"),
    movies=("type", lambda x: (x == "Movie").sum()),
    tv_shows=("type", lambda x: (x == "TV Show").sum())
)

decade_insights

Comparison of movies and TV shows: runtime / seasons

In [ ]:
movie_runtime = prime[prime["type"] == "Movie"]["duration_value"].describe().round(1)
season_count = prime[prime["type"] == "TV Show"]["duration_value"].value_counts().sort_index()

print("Movie runtime (minutes):")
print(movie_runtime)
print()
print("TV shows by number of seasons:")
print(season_count)

Titles with and without director information

In [ ]:
prime["has_director"] = prime["director"].notna().map({True: "Director listed", False: "Director missing"})

prime.groupby("has_director").agg(
    number_of_titles=("title", "count"),
    average_release_year=("release_year", "mean")
).round(2)